# Dark Pattern Detector 🕵️
**Detecting manipulative UI text on shopping websites using NLP**

Dark patterns are design tricks that push users into decisions they did not intend: fake urgency ("Sale ends in 04:59"),
fake scarcity ("Only 2 left!"), social proof ("15 people are viewing this"), confirmshaming ("No thanks, I hate saving money"), etc.
India's Central Consumer Protection Authority (CCPA) banned 13 such patterns in its 2023 guidelines.

This notebook trains and compares three models that classify a piece of website text as **dark pattern / not dark pattern**:
1. TF-IDF + Logistic Regression (classical ML baseline)
2. TF-IDF + Linear SVM (classical ML baseline)
3. Fine-tuned **DistilBERT** (deep learning / transformer)

It also trains a **category classifier** (Urgency, Scarcity, Social Proof, Misdirection, Other) and ends with a **live web demo**.

### How to run
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all` (takes ~10 minutes)

## 1. Setup

In [ ]:
!pip install -q gradio

In [ ]:
import os, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)
results = {}

## 2. Dataset
We use the public dark-pattern dataset from Yada et al. (2022), built from the Princeton *"Dark Patterns at Scale"* study
(Mathur et al., 2019), which crawled ~11K shopping websites. It contains 1,178 dark-pattern texts and 1,178 normal texts.

In [ ]:
URL = "https://raw.githubusercontent.com/yamanalab/ec-darkpattern/master/dataset/dataset.tsv"
df = pd.read_csv(URL, sep="\t").dropna(subset=["text"])
print(df.shape)
df[["text", "label", "Pattern Category"]].sample(8, random_state=1)

In [ ]:
print(df["label"].value_counts(), "\n")
counts = df["Pattern Category"].value_counts()
print(counts)

plt.figure(figsize=(7, 3.5))
counts.plot(kind="barh", color="#4C72B0")
plt.gca().invert_yaxis()
plt.xlabel("Number of texts"); plt.title("Dataset: texts per category")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/dataset_distribution.png", dpi=200); plt.show()

The three rarest categories (Obstruction, Sneaking, Forced Action) have very few examples, so we merge them into **Other**.
We use a stratified 80/20 train/test split.

In [ ]:
df["category"] = df["Pattern Category"].replace(
    {"Obstruction": "Other", "Sneaking": "Other", "Forced Action": "Other"})
train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["category"])
print("Train:", len(train_df), " Test:", len(test_df))

## 3. Evaluation helper

In [ ]:
def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")
    results[name] = dict(accuracy=acc, precision=p, recall=r, f1=f)
    print(f"{name}\n  Accuracy={acc:.4f}  Precision={p:.4f}  Recall={r:.4f}  F1={f:.4f}")
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=["Not dark", "Dark"]).plot(
        cmap="Blues", colorbar=False)
    plt.title(name); plt.tight_layout()
    fname = name.lower().replace(" + ", "_").replace(" ", "_").replace("-", "").replace("(", "").replace(")", "")
    plt.savefig(f"{FIG_DIR}/cm_{fname}.png", dpi=200); plt.show()

## 4. Baselines: TF-IDF + classical ML
TF-IDF turns each text into a vector of word and word-pair (bigram) weights. A linear classifier then learns which words signal a dark pattern.

In [ ]:
def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, lowercase=True)

lr_model = make_pipeline(tfidf(), LogisticRegression(max_iter=1000, C=10))
lr_model.fit(train_df["text"], train_df["label"])
evaluate("TF-IDF + Logistic Regression", test_df["label"], lr_model.predict(test_df["text"]))

svm_model = make_pipeline(tfidf(), LinearSVC(C=1.0))
svm_model.fit(train_df["text"], train_df["label"])
evaluate("TF-IDF + Linear SVM", test_df["label"], svm_model.predict(test_df["text"]))

**Which words matter most?** The Logistic Regression weights show what the model learned:

In [ ]:
vec, clf = lr_model.named_steps["tfidfvectorizer"], lr_model.named_steps["logisticregression"]
words = np.array(vec.get_feature_names_out())
order = np.argsort(clf.coef_[0])
print("Top DARK-pattern words:     ", ", ".join(words[order[-20:]][::-1]))
print("Top NOT-dark-pattern words: ", ", ".join(words[order[:20]]))

## 5. Category classifier
For texts that are dark patterns, a second TF-IDF + Logistic Regression model predicts the **type** of dark pattern.

In [ ]:
dark_tr, dark_te = train_df[train_df.label == 1], test_df[test_df.label == 1]
cat_model = make_pipeline(tfidf(), LogisticRegression(max_iter=1000, C=10, class_weight="balanced"))
cat_model.fit(dark_tr["text"], dark_tr["category"])
cat_pred = cat_model.predict(dark_te["text"])
print(classification_report(dark_te["category"], cat_pred, digits=3, zero_division=0))
rep = classification_report(dark_te["category"], cat_pred, output_dict=True, zero_division=0)
results["Category classifier"] = {"accuracy": rep["accuracy"], "macro_f1": rep["macro avg"]["f1-score"]}

labels = sorted(dark_te["category"].unique())
ConfusionMatrixDisplay(confusion_matrix(dark_te["category"], cat_pred, labels=labels), display_labels=labels).plot(
    cmap="Oranges", colorbar=False, xticks_rotation=30)
plt.title("Category classifier (TF-IDF + LR)"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/cm_category.png", dpi=200); plt.show()

## 6. Deep learning: fine-tuning DistilBERT
DistilBERT is a smaller, faster version of BERT (40% smaller, ~97% of BERT's language understanding).
Unlike TF-IDF, it reads the **context** of the whole sentence. We add a classification head and fine-tune it for 3 epochs.
We hold out 10% of the training set as a validation set to monitor training.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
EPOCHS, BATCH, LR, MAX_LEN = 3, 16, 2e-5, 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_train, bert_val = train_test_split(train_df, test_size=0.1, random_state=SEED, stratify=train_df["label"])

def make_loader(frame, shuffle):
    enc = tokenizer(list(frame["text"]), truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(frame["label"].values))
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle)

train_loader = make_loader(bert_train, True)
val_loader = make_loader(bert_val, False)
test_loader = make_loader(test_df, False)

In [ ]:
bert = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
optimizer = torch.optim.AdamW(bert.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, EPOCHS * len(train_loader))

@torch.no_grad()
def predict_loader(loader):
    bert.eval(); preds, probs = [], []
    for ids, mask, _ in loader:
        logits = bert(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
        p = torch.softmax(logits, dim=-1)[:, 1].cpu()
        probs += p.tolist(); preds += (p > 0.5).long().tolist()
    return np.array(preds), np.array(probs)

history = []
for epoch in range(EPOCHS):
    bert.train(); total = 0
    for ids, mask, y in train_loader:
        optimizer.zero_grad()
        loss = bert(input_ids=ids.to(device), attention_mask=mask.to(device), labels=y.to(device)).loss
        loss.backward(); optimizer.step(); scheduler.step()
        total += loss.item()
    val_acc = accuracy_score(bert_val["label"], predict_loader(val_loader)[0])
    history.append((total / len(train_loader), val_acc))
    print(f"Epoch {epoch+1}/{EPOCHS}  train loss={history[-1][0]:.4f}  val accuracy={val_acc:.4f}")

In [ ]:
bert_pred, bert_prob = predict_loader(test_loader)
evaluate("DistilBERT (fine-tuned)", test_df["label"], bert_pred)

## 7. Model comparison

In [ ]:
binary = {k: v for k, v in results.items() if k != "Category classifier"}
table = pd.DataFrame(binary).T[["accuracy", "precision", "recall", "f1"]]
display(table.style.format("{:.4f}").highlight_max(color="#c6efce"))

ax = table.plot(kind="bar", figsize=(8, 4), ylim=(0.85, 1.0), rot=0, width=0.75)
ax.set_title("Model comparison on the test set"); ax.set_ylabel("Score"); ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/model_comparison.png", dpi=200); plt.show()

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

## 8. Error analysis: what does DistilBERT get wrong?

In [ ]:
errors = test_df.assign(pred=bert_pred, prob_dark=bert_prob.round(3))
errors = errors[errors.pred != errors.label][["text", "label", "pred", "prob_dark", "Pattern Category"]]
print(len(errors), "mistakes out of", len(test_df))
pd.set_option("display.max_colwidth", 120)
errors.head(15)

## 9. Try it on your own sentences

In [ ]:
CATEGORY_INFO = {
    "Urgency": "Creates time pressure (countdowns, 'ends soon')",
    "Scarcity": "Claims limited stock or high demand",
    "Social Proof": "Uses other people's activity to pressure you",
    "Misdirection": "Steers your choice using wording, guilt or visuals",
    "Other": "Obstruction / sneaking / forced action",
}

@torch.no_grad()
def detect(text):
    bert.eval()
    enc = tokenizer([text], truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(device)
    p_dark = torch.softmax(bert(**enc).logits, dim=-1)[0, 1].item()
    category = cat_model.predict([text])[0] if p_dark > 0.5 else None
    return p_dark, category

examples = [
    "Only 2 left in stock - order soon!",
    "Hurry! Deal ends in 04:59",
    "27 people bought this in the last hour",
    "No thanks, I don't like saving money",
    "Rahul from Delhi just purchased this item",
    "Add to cart",
    "Free delivery on orders above Rs. 499",
    "Size chart",
]
for t in examples:
    p, c = detect(t)
    print(f"{'⚠️  DARK' if c else '✅ OK  '}  {p:5.2f}  {c or '':13s}  {t}")

## 10. Live demo 🌐
Paste text copied from any shopping website, **one snippet per line**. Each line is highlighted with its dark-pattern category.
`share=True` gives a public link you can open on your phone or show in class.

In [ ]:
import gradio as gr

def analyze(page_text):
    spans, rows = [], []
    for line in [l.strip() for l in page_text.splitlines() if l.strip()]:
        p, c = detect(line)
        spans += [(line, c), ("\n", None)]
        rows.append([line, "Dark pattern" if c else "OK", c or "-", round(p, 3)])
    flagged = sum(r[1] == "Dark pattern" for r in rows)
    summary = f"### ⚠️ {flagged} of {len(rows)} snippets look like dark patterns" if flagged else "### ✅ No dark patterns found"
    return spans, rows, summary

demo = gr.Interface(
    fn=analyze,
    inputs=gr.Textbox(lines=10, label="Website text (one snippet per line)", value="\n".join(examples)),
    outputs=[gr.HighlightedText(label="Highlighted", combine_adjacent=False),
             gr.Dataframe(headers=["Text", "Verdict", "Category", "P(dark)"], label="Details"),
             gr.Markdown()],
    title="🕵️ Dark Pattern Detector",
    description="DistilBERT detects manipulative text; a TF-IDF classifier names the type.",
    flagging_mode="never",
)
demo.launch(share=True)

## 11. Download figures and results for the report

In [ ]:
!zip -qr report_assets.zip figures results.json
from google.colab import files
files.download("report_assets.zip")